# 📊 TravelMate - Intent Model & NLP Evaluation
### Machine Learning & Semantic Retrieval Evaluation

This notebook presents the evaluation of:
1. **TF-IDF + Logistic Regression Intent Classifier** across 26 travel intents.
2. **Evaluation Metrics:** Accuracy, Precision, Recall, and F1-Score.
3. **Semantic Similarity Testing:** Embedding retrieval with `all-MiniLM-L6-v2`.
4. **End-to-End Test Suite:** Evaluating unseen natural language queries.



In [ ]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np

# Add src to path
sys.path.append(os.path.abspath(".."))
from src.preprocessing import clean_text_for_ml, extract_tokens
from src.semantic_search import SemanticSearchEngine


## 1. Load Training Data and Evaluation Metrics


In [ ]:
MODEL_DIR = os.path.join("..", "models")
metrics_path = os.path.join(MODEL_DIR, "intent_metrics.json")

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

print(f"Model: {metrics.get('model_type')}")
print(f"Overall Accuracy: {metrics.get('accuracy') * 100:.2f}%")
print(f"Macro Precision:  {metrics.get('macro_precision') * 100:.2f}%")
print(f"Macro Recall:     {metrics.get('macro_recall') * 100:.2f}%")
print(f"Macro F1-Score:   {metrics.get('macro_f1') * 100:.2f}%")
print(f"Total Intents:    {metrics.get('num_classes')}")


In [ ]:
# Intent-wise Detailed Report
report_df = pd.DataFrame(metrics['detailed_report']).transpose()
report_df.head(15)


## 2. Intent Classifier Inference on Unseen Queries


In [ ]:
classifier = joblib.load(os.path.join(MODEL_DIR, "intent_classifier.pkl"))
vectorizer = joblib.load(os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl"))

test_queries = [
    "I have ₹8000 and 3 days from Mumbai, where should I go?",
    "What are the best attractions to see in Jaipur?",
    "How can I reach Goa from Mumbai?",
    "What authentic food should I eat in Hyderabad?",
    "Which is better for a weekend, Lonavala or Matheran?",
    "What should I pack for a monsoon trek?",
    "Plan a 4-day itinerary for Rajasthan",
    "I want somewhere quiet and peaceful with nature"
]

results = []
for q in test_queries:
    cleaned = clean_text_for_ml(q)
    vec = vectorizer.transform([cleaned])
    pred = classifier.predict(vec)[0]
    probs = classifier.predict_proba(vec)[0]
    conf = np.max(probs)
    results.append({"Query": q, "Predicted Intent": pred, "Confidence": f"{conf:.2%}"})

pd.DataFrame(results)


## 3. Semantic Search Vector Similarity Evaluation


In [ ]:
engine = SemanticSearchEngine()
engine.initialize()

print("Status:", engine.get_status())


In [ ]:
query = "I want a quiet place away from crowds with beautiful nature and photography"
matches = engine.search_destinations(query, top_k=5)

print(f"Query: '{query}'\n")
print(f"{'Destination':<20} | {'Score':<8} | {'Category':<15} | {'State'}")
print("-" * 65)
for m in matches:
    print(f"{m['destination']:<20} | {m['similarity_score']:.4f}   | {m.get('category',''):<15} | {m.get('state','')}")


## 4. Multi-Constraint Recommendation Verification


In [ ]:
from src.recommender import RecommendationEngine

recommender = RecommendationEngine()
recommender.initialize()

# Complex unseen query: 2 days, ₹7000, nature/peaceful, low crowd, origin Mumbai
recs = recommender.recommend(
    query="I'm travelling from Mumbai for 2 days. Budget around 7000, love nature, peaceful, not crowded",
    entities={"duration": 2, "budget": 7000.0, "origin": "Mumbai"},
    preferences=["nature", "peaceful", "photography"],
    top_k=3
)

for i, r in enumerate(recs, 1):
    print(f"{i}. {r['destination']} (Score: {r['relevance_score']:.3f})")
    print(f"   Category: {r['category']} | Best Season: {r['best_season']}")
    for exp in r['explanations']:
        print(f"   {exp}")
    print()
